<a href="https://colab.research.google.com/github/BenFreilichUVA/walmart-hurricane-project/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, we need to clone the repo so we can actually pull the data and work on the project. Then, we need to load the raw data and put it into pandas DataFrames so the data can be manipulated and changed around. There are five raw files in the data folder that need to be converted.

'walmart_transactions.csv', turned into 'transactions'

'store_locations.csv', turned into 'stores'

'product_catalog.csv', turned into 'products'

'distribution_centers.csv', turned into 'centers'

'inventory_and_sales_db', turned into 'daily_sales' and 'inventory'




In [1]:
!rm -rf /content/walmart-hurricane-project
!git clone https://github.com/BenFreilichUVA/walmart-hurricane-project.git /content/walmart-hurricane-project

from pathlib import Path

def check_repo(root):
    root = Path(root)
    needed = ['CHARTER.md', 'README.md', 'data', 'notebooks/team_notebook.ipynb', 'notebooks/scratch']
    ok = True
    for item in needed:
        found = (root / item).exists()
        print('PASS' if found else 'FAIL', '-', item)
        ok = ok and found
    print()
    print('repo ready:', ok)
    return ok

MY_REPO = '/content/walmart-hurricane-project'
check_repo(MY_REPO)

Cloning into '/content/walmart-hurricane-project'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 34 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 885.95 KiB | 5.79 MiB/s, done.
Resolving deltas: 100% (9/9), done.
PASS - CHARTER.md
PASS - README.md
PASS - data
PASS - notebooks/team_notebook.ipynb
PASS - notebooks/scratch

repo ready: True


True

In [2]:
DATA = Path(MY_REPO) / 'data'

import pandas as pd
import sqlite3
transactions = pd.read_csv(DATA / 'walmart_transactions.csv')
stores       = pd.read_csv(DATA / 'store_locations.csv')
products     = pd.read_csv(DATA / 'product_catalog.csv')
dcs          = pd.read_csv(DATA / 'distribution_centers.csv')

conn = sqlite3.connect(DATA / 'inventory_and_sales.db')

daily_sales = pd.read_sql('SELECT * FROM daily_sales_summary', conn)
inventory   = pd.read_sql('SELECT * FROM inventory_levels', conn)

conn.close()

print('transactions', transactions.shape)
print('stores      ', stores.shape)
print('products    ', products.shape)
print('dcs         ', dcs.shape)
print('daily_sales', daily_sales.shape)
print('inventory   ', inventory.shape)

decision_log = {}


transactions (25562, 8)
stores       (14, 8)
products     (45, 7)
dcs          (5, 8)
daily_sales (3600, 5)
inventory    (4320, 6)


Next, cleaning has to be done. This will take a lot of different steps, and the assignment specifically asks to create a log of each change made.

In [3]:
clean = transactions.copy()

# Fixing the Store IDS for the transaction DataFrame
before = clean['store_id'].nunique()
clean['store_id'] = (clean['store_id']
                     .str.strip()
                     .str.upper()
                     .str.replace(r'^FL[-_]?', 'FL-', regex=True))
after = clean['store_id'].nunique()
# Assertions to ensure this step properly worked
assert clean['store_id'].nunique() == 12, 'expected 12 stores'
assert clean['store_id'].str.fullmatch(r'FL-\d{3}').all(), 'some IDs are not in FL-### format'
assert len(clean) == len(transactions), 'cleaning should not drop rows'
decision_log['transactions: store_id'] = f'Standardized store IDs to FL-###, removing unneccesary spacing and uppercasing ({before} spellings condensed to {after} spellings).'
# Fixing the Categories for the transaction DataFrame
before = clean['category'].nunique()
category_map = {
    'emergency': 'Emergency', 'emergency supplies': 'Emergency', 'grocery': 'Grocery', 'groceries': 'Grocery',
    'electronics': 'Electronics', 'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'household': 'Household', 'household essentials': 'Household', 'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast', 'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'snacks': 'Snacks', 'health': 'Health', 'health & wellness': 'Health',
}
clean['category'] = clean['category'].str.strip().str.lower().map(category_map)
after = clean['category'].nunique()
#Assertions to ensure this step properly worked
assert clean['category'].isna().sum() == 0, 'some categories did not map'
assert set(clean['category']) == set(daily_sales['category']), 'categories do not match the database'
decision_log['transactions: category'] = f'Mapped category names down to the 10 used in the database ({before} spellings condensed to {after} spellings).'

clean['category'].value_counts()

# Fixing the unit_price column to not be stored as text, and to instead be stored as a number
dollar_rows = clean['unit_price'].astype(str).str.contains('$', regex=False).sum()
before_type = clean['unit_price'].dtype

clean['unit_price'] = pd.to_numeric(
    clean['unit_price'].str.replace('$', '', regex=False).str.strip(),
    errors='coerce')

#Assertions to ensure this step properly worked
assert clean['unit_price'].isna().sum() == 0, 'some prices did not convert'
assert (clean['unit_price'] > 0).all(), 'found a zero or negative price'
decision_log['transactions: unit_price'] = 'Removed $ from 3,029 prices and converted the column from text to numbers'
decision_log






{'transactions: store_id': 'Standardized store IDs to FL-###, removing unneccesary spacing and uppercasing (60 spellings condensed to 12 spellings).',
 'transactions: category': 'Mapped category names down to the 10 used in the database (37 spellings condensed to 10 spellings).',
 'transactions: unit_price': 'Removed $ from 3,029 prices and converted the column from text to numbers'}